# working with text data

this notebook implements the text-processing pipeline needed to prepare training data for a gpt-style language model

In [1]:
from importlib.metadata import version
from pathlib import Path
import re

import requests
import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset

print("pytorch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))
print("working directory:", Path.cwd())

pytorch version: 2.13.0
tiktoken version: 0.14.0
working directory: /Users/nabigchaudhry/Projects/build-llm-from-scratch/notebooks


In [6]:
# download and read text

data_directory = Path("data")
data_directory.mkdir(exist_ok=True)

text_path = (data_directory / "the-verdict.txt")

if not text_path.exists():
    text_url = ("https://raw.githubusercontent.com/"
        "rasbt/LLMs-from-scratch/main/"
        "ch02/01_main-chapter-code/"
        "the-verdict.txt")
    response = requests.get(text_url, timeout=30)
    response.raise_for_status()
    text_path.write_bytes(response.content)

raw_text = text_path.read_text(encoding="utf-8")

print("number of characters:", len(raw_text))
print("first 100 characters:", raw_text[:100])

number of characters: 20479
first 100 characters: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [7]:
# experiment with tokenization

sample_text = "Hello, world. Is this-- a text?"

token_pattern = (r"""([,.:;?_!"()']|--|\s)""")

sample_tokens = re.split(token_pattern, sample_text)

sample_tokens = [token.strip() for token in sample_tokens if token.strip()]

print(sample_tokens)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'text', '?']


In [11]:
# use GPT-2 byte pair encoding

bpe_tokenizer = (
    tiktoken.get_encoding("gpt2")
)

sample_text = (
    "Hello, do you like tea? "
    "<|endoftext|> "
    "someunknownPlace"
)

bpe_ids = bpe_tokenizer.encode(
    sample_text,
    allowed_special={"<|endoftext|>"}
)

bpe_tokens = [bpe_tokenizer.decode([token_id]) for token_id in bpe_ids]

print("token_ids:", bpe_ids)
print("tokens:", bpe_tokens)
print("reconstructed text:", bpe_tokenizer.decode(bpe_ids))

token_ids: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 617, 34680, 27271]
tokens: ['Hello', ',', ' do', ' you', ' like', ' tea', '?', ' ', '<|endoftext|>', ' some', 'unknown', 'Place']
reconstructed text: Hello, do you like tea? <|endoftext|> someunknownPlace


In [12]:
# create one next-token example

encoded_text = bpe_tokenizer.encode(raw_text)

context_length = 4

input_ids = encoded_text[50:50 + context_length]

target_ids = encoded_text[51:51 + context_length]

print("input_ids:", input_ids)
print("target_ids:", target_ids)

print("input_text:", bpe_tokenizer.decode(input_ids))
print("target_text:", bpe_tokenizer.decode(target_ids))

input_ids: [290, 4920, 2241, 287]
target_ids: [4920, 2241, 287, 257]
input_text:  and established himself in
target_text:  established himself in a


In [13]:
# display each prediction task

for position in range(context_length):

    context_ids = input_ids[:position + 1]

    desired_ids = target_ids[position]

    context_text = bpe_tokenizer.decode(context_ids)

    desired_text = bpe_tokenizer.decode([desired_ids])

    print(repr(context_text), "->", repr(desired_text))

' and' -> ' established'
' and established' -> ' himself'
' and established himself' -> ' in'
' and established himself in' -> ' a'


In [16]:
# create the gpt dataset

class gpt_dataset_v1(Dataset):

    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        assert(len(token_ids) > max_length)

        for start_index in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[start_index:start_index + max_length]
            target_chunk = token_ids[start_index + 1:start_index + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return (self.input_ids[index], self.target_ids[index])


In [18]:
# create the data loader

def create_dataloader_v1(
        text,
        batch_size=4,
        max_length=256,
        stride=128,
        shuffle=True,
        drop_last=True,
        num_workers=0):

    tokenizer = (tiktoken.get_encoding("gpt2"))

    dataset = gpt_dataset_v1(text, tokenizer, max_length, stride)

    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers)

    return data_loader

In [28]:
# insepct one batch

context_length = 4

data_loader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=context_length,
    stride=context_length,
    shuffle=False)

inputs, targets = next(iter(data_loader))

print("inputs:",inputs)
print("targets:",targets)
print("input shape:",inputs.shape)
print("targets.shape:",targets.shape)

inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
input shape: torch.Size([8, 4])
targets.shape: torch.Size([8, 4])


In [31]:
# create token embeddings

vocab_size = bpe_tokenizer.n_vocab

embedding_dimensions = 256

torch.manual_seed(123)

token_embedding_layer = (torch.nn.Embedding(vocab_size, embedding_dimensions))

token_embeddings = (token_embedding_layer(inputs))

print("embedding weight shape:",token_embedding_layer.weight.shape)
print("token embedding shape:",token_embeddings.shape)

embedding weight shape: torch.Size([50257, 256])
token embedding shape: torch.Size([8, 4, 256])


In [37]:
# add positional embeddings

position_embedding_layer = (torch.nn.Embedding(context_length, embedding_dimensions))

position_ids = torch.arange(context_length)

position_embeddings = position_embedding_layer(position_ids)

input_embeddings = token_embeddings + position_embeddings

print("position IDs:", position_ids)

print("position embedding shape:",position_embeddings.shape)

print("final input embedding shape:",input_embeddings.shape)


position IDs: tensor([0, 1, 2, 3])
position embedding shape: torch.Size([4, 256])
final input embedding shape: torch.Size([8, 4, 256])
